# Cálculo de Coordenadas Visual

In [1]:
import pandas as pd
import json
from google.colab import files
import folium
from folium.plugins import Draw

!rm -rf /content/sample_data

In [2]:
#@title Mapa Interactivo de Marcadores

m = folium.Map(location=[20.5888, -100.3831], zoom_start=12)

draw = Draw(
    export=True,
    filename='data.geojson',
    draw_options={
        'polyline': False,
        'rectangle': False,
        'circle': False,
        'circlemarker': False,
        'polygon': False
    },
    edit_options={'edit': False}
)
draw.add_to(m)

m

In [ ]:
#@title Subir data.json de Marcadores
!rm -rf /content/*
upload = files.upload()

Saving data.geojson to data.geojson


In [ ]:
#@title Visualización de Marcadores y Coordenadas

with open("data.geojson", "r") as f:
    data = json.load(f)

coords = []
for idx, feat in enumerate(data["features"], start=1):
    coord = feat["geometry"]["coordinates"]
    feat["properties"]["id"] = f"Punto {idx}"   # <- añadir propiedad
    coords.append((f"Punto {idx}", coord))

df = pd.DataFrame(
    [(name, lonlat[0], lonlat[1]) for name, lonlat in coords],
    columns=["ID", "Longitud", "Latitud"]
)
df.index = df.index + 1  # para empezar en 1 en vez de 0

with open("data_labeled.geojson", "w") as f:
    json.dump(data, f, indent=2)

lat_center = sum(c[1][1] for c in coords) / len(coords)
lon_center = sum(c[1][0] for c in coords) / len(coords)
m_view = folium.Map(location=[lat_center, lon_center], zoom_start=12)

# --- 5. Añadir los puntos con numeración en círculos ---
for name, (lon, lat) in coords:
    num = name.split()[-1]  # número del punto
    popup_html = f"<b>{name}</b><br>Latitud: {lat:.6f}<br>Longitud: {lon:.6f}"

    folium.Marker(
        location=[lat, lon],
        popup=popup_html,
        icon=folium.DivIcon(html=f"""
            <div style="
                background-color:#4287f5;
                color:white;
                border-radius:50%;
                width:28px;
                height:28px;
                text-align:center;
                font-size:14pt;
                font-weight:bold;
                line-height:28px;
                border:2px solid black;">
                {num}
            </div>
        """)
    ).add_to(m_view)
df



,ID,Longitud,Latitud
1,Punto 1,-100.411649,20.631861
2,Punto 2,-100.458298,20.695746
3,Punto 3,-100.408344,20.611647


In [ ]:
m_view